# Hành vi đám đông bất thường trên MED (Motion Emotion Dataset)

**Dữ liệu.** MED (Rabiee và cộng sự, AVSS 2016; arXiv:1607.07646):
- 31 video, camera cố định đặt trên cao nhìn xuống sân, 854×480 (có viền đen), 29,97 fps.
- Mỗi khung được gán 1 trong 5 hành vi: **Panic** (hoảng loạn), **Fight** (ẩu đả), **Congestion** (ùn tắc), **Obstacle** (vật cản), **Neutral** (bình thường). Nhãn 0 = "không có gì" (đoạn cuối video), bị loại khỏi đánh giá.
- Mật độ cao hơn UMN, có lớp **ùn tắc**: đây là chỗ mô hình mật độ có thể đóng góp.

**Đặc trưng.** Dùng chung với `anomaly_umn.ipynb`:
- lưới 8×8 với optical flow Farneback, có hoặc không lấy trong mask người;
- mật độ người từ MobileCount + Mask (`mc_mask_f4_partB.pth`);
- 5 đặc trưng toàn khung.

**Hai thí nghiệm:**

| | Bài toán | Giao thức | Chỉ số | So với |
|---|---|---|---|---|
| **A** | Phân loại 5 hành vi (có giám sát) | **Bỏ ra một video** (31 lượt), đúng như bài gốc | Độ chính xác trung bình các lớp | Bài gốc: dense trajectory 38,71%; emotion-based 43,64% |
| **B** | Phát hiện bất thường (không giám sát) | Mỗi video: tối đa 200 khung bình thường đầu để train, phần còn lại để test | AUC theo khung, tổng và theo từng loại | B0 (luật demo), B1, AE; chưa có số công bố |

Ablation đóng góp của mô hình đếm: **flow** (không dùng mô hình 1) → **flow·mask** → **flow·mask+density**.

**Cài đặt Kaggle.**
- Bật **Internet**: tải MED từ Dropbox, khoảng 276 MB.
- Add Input: Model `chinhde/mobilecountmask` (version 1).
- GPU T4.

## 1. Cấu hình

In [ ]:
import copy, json, math, random, re, time, urllib.request, zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score, roc_curve

SEED = 42
MED_URL = 'https://www.dropbox.com/s/kgb6wqeuxud7r5t/Motion_Emotion_Dataset.zip?dl=1'
WORK = Path('/kaggle/working')
MED_DIR = WORK / 'med'
WEIGHT_PATH = Path('/kaggle/input/models/chinhde/mobilecountmask/pytorch/default/1/mc_mask_f4_partB.pth')

CLASSES = {1: 'Panic', 2: 'Fight', 3: 'Congestion', 4: 'Obstacle', 5: 'Neutral'}
NEUTRAL = 5

IW, IH = 960, 416               # ảnh đưa vào MobileCount (sau khi cắt viền đen); mật độ ra 240×104
FW, FH = 320, 136               # ảnh xám để tính optical flow
GRID = (8, 8)
NBINS = 8
WIN = 16                        # khung mỗi cửa sổ thời gian (khoảng 0,5 s)
TRAIN_FRAMES = 200              # thí nghiệm B: tối đa 200 khung bình thường đầu mỗi video để train
VAL_FRAMES = 20                 # 20 khung cuối trong phần train đó làm validation
MIN_TRAIN = 100                 # video có ít hơn 100 khung bình thường ở đầu thì không dùng cho thí nghiệm B
SMOOTH = 15
TOPK = 3
CLS_STRIDE = 8                  # thí nghiệm A: lấy một cửa sổ mỗi 8 khung

AE_EPOCHS, AE_LR, AE_BATCH = 60, 1e-3, 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all()
WORK.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)

## 2. Tải MED và đọc nhãn

- Link Dropbox phải có `?dl=1` để tải thẳng file zip. Giải nén bằng `zipfile` và bỏ thư mục rác `__MACOSX`.
- Nhãn nằm trong file MATLAB `dataset_frames_abnormal_labeling.m`, theo dạng `video01(1,830:1515)=4;`. Notebook đọc file này bằng regex, các lệnh gán áp dụng theo đúng thứ tự trong file.
- Số khung trong nhãn có thể lệch 1–2 so với video thật, nên nhãn được cắt hoặc đệm theo số khung thật.

In [ ]:
MED_DIR.mkdir(parents=True, exist_ok=True)
zip_path = MED_DIR / 'med.zip'
if not zip_path.exists():
    urllib.request.urlretrieve(MED_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(MED_DIR, members=[m for m in z.namelist() if not m.startswith('__MACOSX')])
VIDEOS = sorted(MED_DIR.glob('[0-9][0-9][0-9].mp4'))
assert len(VIDEOS) == 31, len(VIDEOS)

src = next(MED_DIR.rglob('dataset_frames_abnormal_labeling.m')).read_text(encoding='latin-1')
src = re.sub(r'%[^\n]*', '', src)                                       # bỏ chú thích MATLAB
LABELS = {int(v): np.zeros(int(n), np.int8) for v, n in re.findall(r'video(\d+)\s*=\s*zeros\(\s*1\s*,\s*(\d+)\s*\)', src)}
for v, a, b, k in re.findall(r'video(\d+)\(\s*1\s*,\s*(\d+)\s*:\s*(\d+|end)\s*\)\s*=\s*(\d+)', src):
    lab = LABELS[int(v)]
    lab[int(a) - 1:len(lab) if b == 'end' else int(b)] = int(k)
assert sorted(LABELS) == list(range(1, 32))
assert all(set(np.unique(l)) <= {0, 1, 2, 3, 4, 5} for l in LABELS.values())
print({CLASSES.get(k, 'không có gì'): int(sum((l == k).sum() for l in LABELS.values())) for k in range(6)})

## 3. Đọc video: cắt viền đen

Video có viền đen ở trên và dưới. Notebook tìm các dòng tối trên 60 khung đầu của mỗi video rồi cắt bỏ, sau đó tạo:
- ảnh màu 960×416 cho MobileCount;
- ảnh xám 320×136 cho optical flow.

Video được đọc theo từng lô 64 khung và xử lý ngay, nên không phải giữ cả video trong RAM.

In [ ]:
def crop_rows(path, n=60, thr=16):
    cap = cv2.VideoCapture(str(path)); rows = []
    for _ in range(n):
        ok, f = cap.read()
        if not ok:
            break
        rows.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).mean(1))
    cap.release()
    bright = np.flatnonzero(np.max(rows, 0) > thr)
    return int(bright[0]), int(bright[-1]) + 1


def read_batches(path, y0, y1, batch=64):
    # Sinh từng lô (ảnh màu cho MobileCount, ảnh xám cho flow) để không giữ cả video màu trong RAM.
    cap = cv2.VideoCapture(str(path))
    while True:
        col, gry = [], []
        for _ in range(batch):
            ok, f = cap.read()
            if not ok:
                break
            f = f[y0:y1]
            col.append(cv2.resize(f, (IW, IH), interpolation=cv2.INTER_AREA))
            gry.append(cv2.cvtColor(cv2.resize(f, (FW, FH), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY))
        if col:
            yield col, gry
        if len(col) < batch:
            break
    cap.release()


META = []
for p in VIDEOS:
    y0, y1 = crop_rows(p)
    cap = cv2.VideoCapture(str(p)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    META.append(dict(video=int(p.stem), path=p, y0=y0, y1=y1, frames_est=n, label_len=len(LABELS[int(p.stem)])))
META = pd.DataFrame(META)
display(META.drop(columns='path'))

## 4. Mô hình đếm: MobileCount + Mask

In [ ]:
DENSITY_SCALE = 100.0
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class ChainedResidualPool(nn.Module):
    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class MobileCountMask(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        f = torchvision.models.mobilenet_v2(weights=None).features
        self.encoder = nn.ModuleList([f[:4], f[4:7], f[7:14], f[14:18]])
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in (24, 32, 96)])
        self.deep = nn.Sequential(nn.Conv2d(320, channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(channels, 1, 1))
        self.mask_head = nn.Sequential(nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1))

    def forward(self, x):
        feats = []
        for stage in self.encoder:
            x = stage(x); feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        m = torch.sigmoid(self.mask_head(y))
        return self.head(y) * m, m


def load_counter(path):
    if not path.exists():
        raise FileNotFoundError(f'Không thấy {path}: hãy Add Input model chinhde/mobilecountmask (version 1).')
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert 'deep.0.weight' in state and any(k.startswith('mask_head') for k in state), 'checkpoint không phải MobileCount + Mask'
    model = MobileCountMask()
    model.load_state_dict(state)
    print('MobileCount + Mask:', path)
    return model.eval().to(DEVICE)


counter = load_counter(WEIGHT_PATH)

## 5. Trích đặc trưng

| Nhóm | Hình dạng | Nội dung |
|---|---|---|
| `F0` | 8×8×11 | Flow toàn ảnh: tốc độ, histogram 8 hướng, entropy hướng, độ đồng hướng |
| `FM` | 8×8×11 | Như `F0` nhưng có trọng số theo mask người |
| `D` | 8×8×2 | Mật độ người và tỉ lệ mask của ô |
| `G0` | 4 | Toàn khung, không dùng mask: tốc độ, entropy, độ đồng hướng, tỉ lệ ngược chiều |
| `G` | 5 | Toàn khung, trong mask: 4 đại lượng trên và tổng số người |

Kết quả được cache vào `med_feats.npz`.

In [ ]:
GH, GW = GRID
CH, CW = FH // GH, FW // GW


def to_cells(a):
    return a.reshape(GH, CH, GW, CW, *a.shape[2:]).swapaxes(1, 2)


@torch.no_grad()
def run_counter(batch_bgr):
    x = np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in batch_bgr]).astype(np.float32) / 255.0
    x = torch.from_numpy(((x - MEAN) / STD).transpose(0, 3, 1, 2)).to(DEVICE)
    den, mprob = counter(x)
    return den[:, 0].float().cpu().numpy() / DENSITY_SCALE, mprob[:, 0].float().cpu().numpy()


def flow_cell_feats(flow, w):
    mag = np.hypot(flow[..., 0], flow[..., 1])
    ang = np.arctan2(flow[..., 1], flow[..., 0])
    b = ((ang + np.pi) / (2 * np.pi) * NBINS).astype(np.int64) % NBINS
    wm = w * mag
    cw, cwm = to_cells(w).sum((2, 3)), to_cells(wm).sum((2, 3))
    speed = cwm / (cw + 1e-6)
    bc, wmc = to_cells(b), to_cells(wm)
    hist = np.stack([(wmc * (bc == k)).sum((2, 3)) for k in range(NBINS)], -1)
    p = hist / (cwm[..., None] + 1e-6)
    ent = -(p * np.log(p + 1e-9)).sum(-1) / np.log(NBINS)
    vx, vy = to_cells(w * flow[..., 0]).sum((2, 3)), to_cells(w * flow[..., 1]).sum((2, 3))
    coh = np.hypot(vx, vy) / (cwm + 1e-6)
    return np.concatenate([speed[..., None], p, ent[..., None], coh[..., None]], -1).astype(np.float32)


def global_feats(flow, w):
    mag = np.hypot(flow[..., 0], flow[..., 1])
    ang = np.arctan2(flow[..., 1], flow[..., 0])
    wm = w * mag
    tot = wm.sum() + 1e-6
    speed = wm.sum() / (w.sum() + 1e-6)
    b = ((ang + np.pi) / (2 * np.pi) * NBINS).astype(np.int64) % NBINS
    p = np.bincount(b.ravel(), weights=wm.ravel(), minlength=NBINS) / tot
    ent = -(p * np.log(p + 1e-9)).sum() / np.log(NBINS)
    vx, vy = (w * flow[..., 0]).sum(), (w * flow[..., 1]).sum()
    coh = np.hypot(vx, vy) / tot
    dev = np.abs(np.angle(np.exp(1j * (ang - np.arctan2(vy, vx)))))
    return np.array([speed, ent, coh, (wm * (dev > 2 * np.pi / 3)).sum() / tot], np.float32)


FEAT_CACHE = WORK / 'med_feats.npz'
if FEAT_CACHE.exists():
    z = np.load(FEAT_CACHE)
    F0, FM, D, G0, G, video_id, label = (z[k] for k in ('F0', 'FM', 'D', 'G0', 'G', 'video_id', 'label'))
    print('đọc cache', FEAT_CACHE)
else:
    t0 = time.time()
    parts = {k: [] for k in ('F0', 'FM', 'D', 'G0', 'G', 'video_id', 'label')}
    ones = np.ones((FH, FW), np.float32)
    for m in META.itertuples():
        f0, fm, d, g0, g = [], [], [], [], []
        prev = None
        for col, gry in read_batches(m.path, m.y0, m.y1):
            den, mprob = run_counter(col)
            for j in range(len(den)):
                dh, dw = den[j].shape
                mpx = cv2.resize(mprob[j], (FW, FH), interpolation=cv2.INTER_LINEAR)
                d.append(np.stack([den[j].reshape(GH, dh // GH, GW, dw // GW).sum((1, 3)), to_cells(mpx).mean((2, 3))], -1))
                flow = (np.zeros((FH, FW, 2), np.float32) if prev is None else
                        cv2.calcOpticalFlowFarneback(prev, gry[j], None, 0.5, 3, 15, 3, 5, 1.2, 0))
                prev = gry[j]
                f0.append(flow_cell_feats(flow, ones)); fm.append(flow_cell_feats(flow, mpx))
                g0.append(global_feats(flow, ones)); g.append(np.r_[global_feats(flow, mpx), den[j].sum()].astype(np.float32))
        n = len(g)
        lab = np.zeros(n, np.int8); k = min(n, len(LABELS[m.video])); lab[:k] = LABELS[m.video][:k]
        g = np.stack(g)
        for k_, v_ in (('F0', np.stack(f0)), ('FM', np.stack(fm)), ('D', np.stack(d)), ('G0', np.stack(g0)), ('G', g),
                       ('video_id', np.full(n, m.video, np.int16)), ('label', lab)):
            parts[k_].append(v_)
        print(f'  video {m.video:02d}: {n} khung, {g[:, 4].mean():.1f} người/khung · {time.time() - t0:.0f}s')
    F0, FM, D, G0, G, video_id, label = (np.concatenate(parts[k]) for k in ('F0', 'FM', 'D', 'G0', 'G', 'video_id', 'label'))
    np.savez_compressed(FEAT_CACHE, F0=F0, FM=FM, D=D, G0=G0, G=G, video_id=video_id, label=label)
    print(f'trích đặc trưng xong trong {time.time() - t0:.0f}s')

N = len(label)
v_first = pd.Series(np.arange(N)).groupby(video_id).transform('min').to_numpy()
print(N, 'khung;', 'số người trung bình theo hành vi:',
      {CLASSES[k]: round(float(G[label == k, 4].mean()), 1) for k in CLASSES if (label == k).any()})

## 6. Thí nghiệm A: phân loại 5 hành vi, giao thức bỏ ra một video (giống bài gốc)

- **Mẫu:** cửa sổ 16 khung, lấy mỗi 8 khung một cửa sổ. Chỉ giữ cửa sổ có cả 16 khung cùng một nhãn và nhãn khác 0.
- **Đặc trưng của cửa sổ:**
  - với mỗi khung: các đặc trưng toàn khung, cộng với trung bình, max và độ lệch chuẩn qua 64 ô của từng kênh lưới;
  - sau đó lấy trung bình và độ lệch chuẩn theo thời gian trong cửa sổ.
- **Bộ phân loại:** Random Forest 300 cây, cân bằng lớp.
- **Giao thức:** 31 lượt, mỗi lượt để riêng 1 video làm test.
- **Chỉ số:** độ chính xác trung bình các lớp (balanced accuracy), là "average accuracy over all classes" mà bài gốc dùng.

Ba tập đặc trưng cho ablation:

| Tập | Gồm |
|---|---|
| flow | `G0` + thống kê lưới của `F0` (không dùng mô hình đếm) |
| flow·mask | `G[:4]` + thống kê lưới của `FM` |
| flow·mask+density | `G` (có số người) + thống kê lưới của `FM` và `D` |

In [ ]:
def frame_vectors(G_, grid):
    flat = grid.reshape(len(grid), GH * GW, -1)
    return np.concatenate([G_, flat.mean(1), flat.max(1), flat.std(1)], 1).astype(np.float32)


FEATSETS = {
    'flow': frame_vectors(G0, F0),
    'flow·mask': frame_vectors(G[:, :4], FM),
    'flow·mask+density': frame_vectors(G, np.concatenate([FM, D], -1)),
}

ends = np.array([t for t in range(N) if t - WIN + 1 >= v_first[t] and (t - v_first[t]) % CLS_STRIDE == WIN - 1])
same = np.array([(label[t - WIN + 1:t + 1] == label[t]).all() for t in ends])
ends = ends[same & (label[ends] != 0)]
y_cls, g_cls = label[ends], video_id[ends]
print(len(ends), 'cửa sổ;', pd.Series(y_cls).map(CLASSES).value_counts().to_dict())


def window_matrix(V):
    W_ = np.stack([V[t - WIN + 1:t + 1] for t in ends])
    return np.concatenate([W_.mean(1), W_.std(1)], 1)


CLS_PRED = {}
for name, V in FEATSETS.items():
    X = window_matrix(V)
    pred = np.zeros_like(y_cls)
    for v in np.unique(g_cls):
        te = g_cls == v
        rf = RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', min_samples_leaf=2,
                                    n_jobs=-1, random_state=SEED).fit(X[~te], y_cls[~te])
        pred[te] = rf.predict(X[te])
    CLS_PRED[name] = pred
    print(f'{name:>20}: độ chính xác trung bình các lớp = {balanced_accuracy_score(y_cls, pred):.4f}')

In [ ]:
rows = [('Dense trajectory (Rabiee et al., AVSS 2016)', 0.3871, 'bài gốc, đặc trưng cấp thấp'),
        ('Dense trajectory + emotion-based (Rabiee et al.)', 0.4364, 'bài gốc, có dùng nhãn cảm xúc khi train')]
rows += [(f'Đề tài · RF trên {n}', balanced_accuracy_score(y_cls, p), 'notebook này') for n, p in CLS_PRED.items()]
cls_table = pd.DataFrame(rows, columns=['Phương pháp', 'Độ chính xác TB các lớp', 'Nguồn'])
display(cls_table.style.format({'Độ chính xác TB các lớp': '{:.2%}'}))
cls_table.to_csv(WORK / 'med_classification.csv', index=False)

best = max(CLS_PRED, key=lambda n: balanced_accuracy_score(y_cls, CLS_PRED[n]))
labels_order = [1, 2, 3, 4, 5]
cm = confusion_matrix(y_cls, CLS_PRED[best], labels=labels_order, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(5), [CLASSES[k] for k in labels_order], rotation=30); ax.set_yticks(range(5), [CLASSES[k] for k in labels_order])
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{cm[i, j]:.0%}', ha='center', va='center', color='white' if cm[i, j] > .5 else 'black', fontsize=9)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thực tế'); ax.set_title(f'Ma trận nhầm lẫn · {best}')
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig(WORK / 'med_confusion.png', dpi=130); plt.show()

## 7. Thí nghiệm B: phát hiện bất thường không giám sát

- **Train:** với mỗi video, tối đa 200 khung **bình thường liên tiếp ở đầu video** (20 khung cuối trong đó làm validation).
  - Video có ít hơn 100 khung bình thường ở đầu bị loại khỏi thí nghiệm này. Đó là video 30 và 31, vì chúng bắt đầu bằng vật cản.
- **Test:** toàn bộ khung còn lại có nhãn khác 0. Bất thường = Panic, Fight, Congestion hoặc Obstacle.
- **Chuẩn hóa theo từng video,** dùng thống kê của chính các khung train của video đó. Việc này giống bước "học mức bình thường 5 phút đầu" trong demo.
- **Mô hình:** B0 (luật demo), B1 (Isolation Forest trên đặc trưng toàn khung) và 3 biến thể AE. Mỗi mô hình được train chung trên các khung train của mọi video.
- **Chỉ số:**
  - AUC tổng;
  - AUC từng loại (loại đó so với bình thường);
  - AUC trung bình theo video (chỉ tính video có cả hai nhãn).

In [ ]:
split = np.full(N, 'test', dtype=object)
used_videos = []
for v in np.unique(video_id):
    idx = np.flatnonzero(video_id == v)
    lab = label[idx]
    n_norm = int(np.argmax(lab != NEUTRAL)) if (lab != NEUTRAL).any() else len(lab)
    n_tr = min(TRAIN_FRAMES, n_norm)
    if n_tr < MIN_TRAIN:
        split[idx] = 'unused'
        continue
    used_videos.append(int(v))
    split[idx[:n_tr - VAL_FRAMES]] = 'train'
    split[idx[n_tr - VAL_FRAMES:n_tr]] = 'val'
split[(split == 'test') & (label == 0)] = 'unused'
assert ((split == 'train') | (split == 'val')).sum() > 0 and (label[(split == 'train') | (split == 'val')] == NEUTRAL).all()
print('video dùng:', len(used_videos), '| bỏ:', sorted(set(range(1, 32)) - set(used_videos)))
print(pd.crosstab(pd.Series(label).map(lambda k: CLASSES.get(k, 'không có gì')), split))

WIDX = np.maximum(np.arange(N)[:, None] - np.arange(WIN)[::-1][None, :], v_first[:, None])


def normalize_per_video(X):
    # z-score riêng mỗi video theo khung train của nó. X: (N, ...)
    Xn = np.zeros_like(X)
    for v in used_videos:
        idx = np.flatnonzero(video_id == v)
        tr = idx[split[idx] == 'train']
        mu, sd = X[tr].mean(0), X[tr].std(0)
        Xn[idx] = np.clip((X[idx] - mu) / (sd + 1e-3), -10, 10)
    return Xn


def robust_z(x, ref):
    med = np.median(ref)
    return (x - med) / (1.4826 * np.median(np.abs(ref - med)) + 1e-6)


def causal_smooth(score):
    out = np.zeros_like(score)
    for v in np.unique(video_id):
        idx = np.flatnonzero(video_id == v)
        cs = np.cumsum(np.r_[0.0, score[idx]])
        k = np.arange(1, len(idx) + 1); lo = np.maximum(0, k - SMOOTH)
        out[idx] = (cs[k] - cs[lo]) / (k - lo)
    return out

In [ ]:
class GridAE(nn.Module):
    def __init__(self, cin, hid=64, code=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(cin, hid, 3, padding=1), nn.ReLU(inplace=True),
                                 nn.Conv2d(hid, code, 3, padding=1), nn.ReLU(inplace=True))
        self.dec = nn.Sequential(nn.Conv2d(code, hid, 3, padding=1), nn.ReLU(inplace=True),
                                 nn.Conv2d(hid, cin, 3, padding=1))

    def forward(self, x):
        return self.dec(self.enc(x))


def windows(Xt, frames_idx):
    x = Xt[torch.from_numpy(WIDX[frames_idx]).to(DEVICE)]        # (B, WIN, C, GH, GW)
    return x.reshape(x.shape[0], -1, GH, GW)


def train_ae(Xt, name):
    seed_all()
    tr, va = np.flatnonzero(split == 'train'), np.flatnonzero(split == 'val')
    model = GridAE(WIN * Xt.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=AE_LR)
    best, best_state, hist = math.inf, None, []
    for ep in range(AE_EPOCHS):
        model.train()
        perm = np.random.permutation(tr)
        for s in range(0, len(perm), AE_BATCH):
            xb = windows(Xt, perm[s:s + AE_BATCH])
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(np.mean([F.mse_loss(model(xb := windows(Xt, va[s:s + 512])), xb).item() for s in range(0, len(va), 512)]))
        hist.append(vl)
        if vl < best:
            best, best_state = vl, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f'  {name}: val loss tốt nhất {best:.4f} (epoch {int(np.argmin(hist)) + 1})')
    return model, hist


@torch.no_grad()
def ae_score(model, Xt, idx):
    scores, maps = np.zeros(len(idx), np.float32), np.zeros((len(idx), GH, GW), np.float32)
    for s in range(0, len(idx), 512):
        xb = windows(Xt, idx[s:s + 512])
        err = (model(xb) - xb).pow(2).reshape(len(xb), WIN, -1, GH, GW).mean((1, 2))
        maps[s:s + len(xb)] = err.cpu().numpy()
        scores[s:s + len(xb)] = err.flatten(1).topk(TOPK, dim=1).values.mean(1).cpu().numpy()
    return scores, maps


VARIANTS = {'AE-flow': F0, 'AE-flow·mask': FM, 'AE-flow·mask+density': np.concatenate([FM, D], -1)}
use = np.flatnonzero(split != 'unused')
tr = np.flatnonzero(split == 'train')
SCORES, ERRMAPS, AE_HIST = {}, {}, {}

Gn, G0n = normalize_per_video(G), normalize_per_video(G0)
SCORES['B0 luật demo'] = np.minimum(Gn[:, 0], Gn[:, 1])
for name, GG in (('B1 IsolationForest · flow', G0n), ('B1 IsolationForest · flow·mask+count', Gn)):
    iso = IsolationForest(n_estimators=300, random_state=SEED).fit(GG[tr])
    SCORES[name] = -iso.score_samples(GG)

for name, X in VARIANTS.items():
    Xt = torch.from_numpy(np.ascontiguousarray(normalize_per_video(X).transpose(0, 3, 1, 2))).float().to(DEVICE)
    model, hist = train_ae(Xt, name)
    s = np.zeros(N, np.float32)
    s[use], maps = ae_score(model, Xt, use)
    SCORES[name], ERRMAPS[name], AE_HIST[name] = s, maps, hist
    torch.save(model.state_dict(), WORK / f'ae_med_{name.replace("·", "_").replace("+", "_")}.pth')
    del Xt

SMOOTHED = {m: causal_smooth(s.astype(np.float64)) for m, s in SCORES.items()}

In [ ]:
def eer(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return (fpr[i] + 1 - tpr[i]) / 2


test = split == 'test'
y = (label != NEUTRAL).astype(int)
rows = []
for m, s in SMOOTHED.items():
    row = {'Phương pháp': m, 'AUC tổng': roc_auc_score(y[test], s[test]), 'EER tổng': eer(y[test], s[test])}
    for k in (1, 2, 3, 4):
        sel = test & ((label == k) | (label == NEUTRAL))
        if (label[sel] == k).any():
            row[f'AUC {CLASSES[k]}'] = roc_auc_score(label[sel] == k, s[sel])
    per_video = [roc_auc_score(y[test & (video_id == v)], s[test & (video_id == v)])
                 for v in used_videos if len(np.unique(y[test & (video_id == v)])) == 2]
    row['AUC TB theo video'] = np.mean(per_video)
    rows.append(row)
unsup = pd.DataFrame(rows).set_index('Phương pháp')
display(unsup.style.format('{:.3f}').highlight_max(axis=0, subset=[c for c in unsup.columns if 'AUC' in c]))
unsup.to_csv(WORK / 'med_unsupervised.csv')
print('Số video có cả bình thường và bất thường trong tập test:',
      sum(len(np.unique(y[test & (video_id == v)])) == 2 for v in used_videos))

## 8. Hình minh họa

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for m, s in SMOOTHED.items():
    fpr, tpr, _ = roc_curve(y[test], s[test])
    ax.plot(fpr, tpr, label=f'{m} ({unsup.loc[m, "AUC tổng"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=.8)
ax.set_xlabel('Tỉ lệ báo nhầm'); ax.set_ylabel('Tỉ lệ phát hiện'); ax.set_title('ROC trên MED (thí nghiệm B)')
ax.legend(fontsize=7, loc='lower right'); plt.tight_layout(); plt.savefig(WORK / 'med_roc.png', dpi=130); plt.show()

show_v = [v for v in (1, 10, 15, 19) if v in used_videos]
colors = {1: 'tab:red', 2: 'tab:purple', 3: 'tab:orange', 4: 'tab:brown'}
fig, axes = plt.subplots(len(show_v), 1, figsize=(15, 2.6 * len(show_v)))
for ax, v in zip(np.atleast_1d(axes), show_v):
    idx = np.flatnonzero(video_id == v)
    for k, c in colors.items():
        ax.fill_between(np.arange(len(idx)), 0, 1, where=label[idx] == k, color=c, alpha=.18,
                        transform=ax.get_xaxis_transform(), label=CLASSES[k])
    for m in ('B0 luật demo', 'B1 IsolationForest · flow·mask+count', 'AE-flow·mask+density'):
        s = SMOOTHED[m][idx]; ref = SMOOTHED[m][idx][split[idx] == 'train']
        ax.plot(np.clip(robust_z(s, ref), -5, 40), lw=1, label=m)
    ax.set_ylabel(f'video {v:02d}')
handles, labs = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.legend(dict(zip(labs, handles)).values(), dict(zip(labs, handles)).keys(), ncol=7, fontsize=8, loc='upper center')
plt.tight_layout(rect=(0, 0, 1, .95)); plt.savefig(WORK / 'med_timeline.png', dpi=120); plt.show()

## 9. Lưu kết quả

In [ ]:
json.dump(dict(counter_checkpoint=str(WEIGHT_PATH), win=WIN, grid=GRID, cls_stride=CLS_STRIDE,
               train_frames=TRAIN_FRAMES, val_frames=VAL_FRAMES, smooth=SMOOTH, ae_epochs=AE_EPOCHS, seed=SEED,
               used_videos=used_videos,
               classification=cls_table.round(4).to_dict(orient='records'),
               unsupervised=unsup.round(4).reset_index().to_dict(orient='records')),
          open(WORK / 'med_config_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(cls_table.to_string()); print(); print(unsup.round(3).to_string())

## Cách đọc kết quả

- **Thí nghiệm A** so sánh trực tiếp được với bài gốc: cùng 5 lớp, cùng giao thức bỏ ra một video, cùng chỉ số. Bài gốc phân loại theo clip; ở đây đơn vị là cửa sổ 16 khung (khoảng 0,5 giây), nên là bài toán cùng loại.
  - So flow → flow·mask → flow·mask+density để biết mô hình đếm đóng góp bao nhiêu, nhất là ở lớp **Congestion**.
- **Thí nghiệm B** chưa có số công bố để so. Dùng để:
  - so luật demo (B0) với mô hình học được;
  - xem AUC từng loại: loại nào dễ, loại nào khó phát hiện khi không có nhãn.
- **Giới hạn:**
  - MED chỉ có vài bối cảnh quay, và người tham gia diễn theo kịch bản.
  - Nhãn từng khung đọc từ file `.m` của tác giả, có thể lệch 1–2 khung ở ranh giới.